# Bronze Ingestion — StatFin Data from Azure ADLS
Reads raw CSV files from Azure Blob Storage (via SAS token) and writes them as Delta tables into the Unity Catalog bronze schema.

| Table | Source file | Description |
|---|---|---|
| `bronze_statfin_bankruptcies` | `13ff.csv` | Bankruptcies by municipality and industry |
| `bronze_statfin_enterprise_establishments` | `13wz.csv` | Enterprise establishments by municipality |
| `bronze_statfin_population` | `12au.csv` | Population and deaths by municipality |

## Cell 1 — Configuration
Secrets are read from the Databricks secret scope `academy-case`. Never hardcode tokens here.  
To update: `databricks secrets put-secret academy-case sas_token --string-value "?sv=..."`

In [ ]:
# ── Secrets (never hardcode) ──────────────────────────────────────────────────
sas             = dbutils.secrets.get(scope="academy-case", key="sas_token")
storage_account = "dataacademyandreasandrsa"
container       = "raw-data"

# ── Unity Catalog target ──────────────────────────────────────────────────────
catalog         = "data_academy_fi_06"
bronze_schema   = "andreas_statfin_bronze"

# ── Source file mapping: logical_name -> (adls_folder, filename, bronze_table) ─
sources = {
    "bankruptcies": (
        "bankruptcies",
        "13ff.csv",
        "bronze_statfin_bankruptcies",
    ),
    "enterprise_establishments": (
        "enterprise-establishments",
        "13wz.csv",
        "bronze_statfin_enterprise_establishments",
    ),
    "population": (
        "population",
        "12au.csv",
        "bronze_statfin_population",
    ),
}

# ── Derived base path ─────────────────────────────────────────────────────────
base = f"wasbs://{container}@{storage_account}.blob.core.windows.net"
print(f"Base URL : {base}")
print(f"Target   : {catalog}.{bronze_schema}")

## Cell 2 — Connectivity Check
Lists files in each ADLS folder to verify the SAS token and folder structure are correct before loading.

In [ ]:
all_ok = True
for name, (folder, filename, _) in sources.items():
    url = f"{base}/{folder}/{sas}"
    try:
        files = dbutils.fs.ls(url)
        match = [f for f in files if f.name == filename]
        status = "✅ found" if match else "⚠️  folder ok but file not found"
        print(f"{name:30s} {status}  ({url[:60]}...)")
    except Exception as e:
        print(f"{name:30s} ❌ ERROR: {e}")
        all_ok = False

if not all_ok:
    raise Exception("Connectivity check failed — fix errors above before continuing.")

## Cell 3 — Read CSVs into Spark DataFrames
Reads each CSV with header inference and adds two metadata columns:
- `_source_path` — the ADLS path the file was read from
- `_load_ts` — timestamp when this run loaded the data

In [ ]:
from pyspark.sql import functions as F

dataframes = {}

for name, (folder, filename, table) in sources.items():
    path = f"{base}/{folder}/{filename}{sas}"
    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .option("quote", '"')
        .option("escape", '"')
        .csv(path)
        .withColumn("_source_path", F.lit(f"{folder}/{filename}"))
        .withColumn("_load_ts", F.current_timestamp())
    )
    dataframes[name] = df
    print(f"✅ {name:30s}  rows={df.count():>7,}  cols={len(df.columns)}")

print("\nAll CSVs loaded.")

## Cell 4 — Preview Schemas
Quick check to confirm column names and types look correct before writing to Delta.

In [ ]:
for name, df in dataframes.items():
    print(f"\n{'─'*60}")
    print(f"  {name}")
    print(f"{'─'*60}")
    df.printSchema()
    display(df.limit(3))

## Cell 5 — Write to Bronze Delta Tables
Writes each DataFrame as a managed Delta table in Unity Catalog using `CREATE OR REPLACE`.
This is a full refresh — safe to re-run at any time.

In [ ]:
spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"USE SCHEMA {bronze_schema}")

for name, (_, _, table) in sources.items():
    df = dataframes[name]
    full_table = f"{catalog}.{bronze_schema}.{table}"
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(full_table)
    )
    count = spark.table(full_table).count()
    print(f"✅ Written {count:>7,} rows → {full_table}")

print("\nBronze ingestion complete.")

## Cell 6 — Verify Tables in Unity Catalog
Confirms all three tables exist and shows row counts and column lists.

In [ ]:
print(f"{'Table':<50} {'Rows':>8}  Columns")
print("─" * 90)
for name, (_, _, table) in sources.items():
    full_table = f"{catalog}.{bronze_schema}.{table}"
    df = spark.table(full_table)
    cols = [c for c in df.columns if not c.startswith("_")]
    print(f"{full_table:<50} {df.count():>8,}  {', '.join(cols)}")